# AI-Driven Cloud Auto-Scaling — Model Training Notebook

This notebook walks through the complete ML pipeline:
1. Load the workload dataset
2. Explore & visualise the data
3. Preprocess & engineer features
4. Train a Linear Regression model
5. Evaluate performance (R², MAE)
6. Save the trained model
7. Test predictions on sample inputs

---
## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

print("All libraries imported successfully!")

---
## 2. Load the Dataset

In [ ]:
# Load the workload dataset
DATASET_PATH = os.path.join("..", "dataset", "workload_dataset.csv")
df = pd.read_csv(DATASET_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 10 rows:")
df.head(10)

In [ ]:
# Basic statistics
print("Dataset Statistics:")
df.describe()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

---
## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Plot all metrics over time
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Requests over time
axes[0].plot(df['timestamp'], df['requests'], color='#2196F3', linewidth=2)
axes[0].set_ylabel('Requests / sec')
axes[0].set_title('Workload Traffic Pattern Over Time')
axes[0].grid(True, alpha=0.3)

# CPU usage over time
axes[1].plot(df['timestamp'], df['cpu_usage'], color='#FF5722', linewidth=2)
axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.5, label='SLA Threshold (80%)')
axes[1].set_ylabel('CPU Usage (%)')
axes[1].set_title('CPU Utilisation Over Time')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Memory usage over time
axes[2].plot(df['timestamp'], df['memory_usage'], color='#4CAF50', linewidth=2)
axes[2].set_ylabel('Memory Usage (%)')
axes[2].set_xlabel('Timestamp')
axes[2].set_title('Memory Utilisation Over Time')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
correlation = df.corr()
print("Correlation Matrix:")
print(correlation)
print()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(correlation, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation.columns)))
ax.set_yticks(range(len(correlation.columns)))
ax.set_xticklabels(correlation.columns, rotation=45)
ax.set_yticklabels(correlation.columns)

# Add correlation values as text
for i in range(len(correlation)):
    for j in range(len(correlation)):
        ax.text(j, i, f'{correlation.iloc[i, j]:.3f}',
                ha='center', va='center', fontweight='bold')

plt.colorbar(im)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: each feature vs requests
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(df['cpu_usage'], df['requests'], alpha=0.6, color='#FF5722')
axes[0].set_xlabel('CPU Usage (%)')
axes[0].set_ylabel('Requests / sec')
axes[0].set_title('Requests vs CPU Usage')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df['memory_usage'], df['requests'], alpha=0.6, color='#4CAF50')
axes[1].set_xlabel('Memory Usage (%)')
axes[1].set_ylabel('Requests / sec')
axes[1].set_title('Requests vs Memory Usage')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Data Preprocessing

In [ ]:
# Step 1: Handle missing values (forward fill + drop)
df_clean = df.ffill().dropna().reset_index(drop=True)
print(f"After cleaning — shape: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")

In [ ]:
# Step 2: Feature Engineering
# We predict the NEXT time step's request count
# Features: current [timestamp, requests, cpu_usage]
# Target:   next row's [requests] (shifted by -1)

FEATURE_COLUMNS = ['timestamp', 'requests', 'cpu_usage']
TARGET_COLUMN = 'requests'

df_clean['target'] = df_clean[TARGET_COLUMN].shift(-1)  # next row's requests
df_clean = df_clean.dropna()  # drop last row (no next value)

print(f"After feature engineering — shape: {df_clean.shape}")
print(f"\nSample rows (features + target):")
df_clean[FEATURE_COLUMNS + ['target']].head(10)

In [ ]:
# Step 3: Separate features (X) and target (y)
X = df_clean[FEATURE_COLUMNS].values
y = df_clean['target'].values

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape:  {y.shape}")
print(f"\nFeatures: {FEATURE_COLUMNS}")
print(f"Target:   next time-step requests")

---
## 5. Train-Test Split

In [ ]:
# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Split ratio:  {X_train.shape[0]}/{X_test.shape[0]} = 80%/20%")

---
## 6. Train the Model

In [ ]:
# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nModel coefficients:")
for feature, coef in zip(FEATURE_COLUMNS, model.coef_):
    print(f"  {feature:>15}: {coef:.6f}")
print(f"  {'intercept':>15}: {model.intercept_:.6f}")
print(f"\nPrediction formula:")
print(f"  predicted_requests = ({model.coef_[0]:.4f} x timestamp) + "
      f"({model.coef_[1]:.4f} x requests) + ({model.coef_[2]:.4f} x cpu_usage) + "
      f"({model.intercept_:.4f})")

---
## 7. Evaluate the Model

In [ ]:
# Make predictions on test set
y_pred = model.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("       MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"  R-squared (R2):              {r2:.4f}")
print(f"  Mean Absolute Error (MAE):   {mae:.4f}")
print(f"  Mean Squared Error (MSE):    {mse:.4f}")
print(f"  Root Mean Squared Error:     {rmse:.4f}")
print("=" * 50)
print(f"\nInterpretation:")
print(f"  - R2 = {r2:.4f} means the model explains {r2*100:.2f}% of the variance")
print(f"  - MAE = {mae:.2f} means predictions are off by ~{mae:.0f} requests on average")

In [ ]:
# Plot: Actual vs Predicted (test set)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.7, color='#2196F3', edgecolors='white')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect Prediction Line')
axes[0].set_xlabel('Actual Requests')
axes[0].set_ylabel('Predicted Requests')
axes[0].set_title(f'Actual vs Predicted (R² = {r2:.4f})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residuals plot
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.7, color='#FF5722', edgecolors='white')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=1)
axes[1].set_xlabel('Predicted Requests')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residuals Plot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(residuals, bins=15, color='#9C27B0', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Residual (Actual - Predicted)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Prediction Errors')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.4f} (should be close to 0)")
print(f"Std of residuals: {residuals.std():.4f}")

---
## 8. Save the Trained Model

In [ ]:
# Save model using joblib
MODEL_DIR = os.path.join("..", "models")
MODEL_PATH = os.path.join(MODEL_DIR, "trained_model.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(model, MODEL_PATH)

print(f"Model saved to: {os.path.abspath(MODEL_PATH)}")
print(f"File size: {os.path.getsize(MODEL_PATH)} bytes")

In [ ]:
# Verify: reload the model and confirm it works
loaded_model = joblib.load(MODEL_PATH)
y_verify = loaded_model.predict(X_test)
verify_r2 = r2_score(y_test, y_verify)

print(f"Reloaded model R2: {verify_r2:.4f}")
print(f"Original model R2: {r2:.4f}")
print(f"Match: {np.allclose(y_pred, y_verify)}")

---
## 9. Test Predictions on Sample Inputs

In [ ]:
# Define test scenarios
test_scenarios = [
    {"name": "Low Traffic",    "timestamp": 10, "requests": 100, "cpu_usage": 30.0},
    {"name": "Medium Traffic", "timestamp": 50, "requests": 250, "cpu_usage": 65.0},
    {"name": "High Traffic",   "timestamp": 70, "requests": 400, "cpu_usage": 90.0},
    {"name": "Peak Traffic",   "timestamp": 44, "requests": 420, "cpu_usage": 97.0},
    {"name": "Cooldown",       "timestamp": 85, "requests": 180, "cpu_usage": 56.0},
]

print(f"{'Scenario':<18} {'Timestamp':>9} {'Current Reqs':>12} {'CPU %':>7} {'Predicted':>10} {'Containers':>11}")
print("-" * 75)

for s in test_scenarios:
    features = np.array([[s['timestamp'], s['requests'], s['cpu_usage']]])
    prediction = max(0, int(round(loaded_model.predict(features)[0])))
    import math
    containers = math.ceil(prediction / 100)
    print(f"{s['name']:<18} {s['timestamp']:>9} {s['requests']:>12} {s['cpu_usage']:>7.1f} {prediction:>10} {containers:>11}")

In [ ]:
# Full dataset prediction vs actual (visual check)
df_full = pd.read_csv(DATASET_PATH)
df_full['target'] = df_full['requests'].shift(-1)
df_full = df_full.dropna()

X_full = df_full[FEATURE_COLUMNS].values
y_full = df_full['target'].values
y_full_pred = loaded_model.predict(X_full)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_full['timestamp'], y_full, label='Actual Next Requests',
        linewidth=2, color='#2196F3')
ax.plot(df_full['timestamp'], y_full_pred, label='Predicted Next Requests',
        linewidth=2, linestyle='--', color='#FF5722')
ax.set_xlabel('Timestamp')
ax.set_ylabel('Requests / sec')
ax.set_title('Full Dataset: Actual vs Predicted Workload')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

full_r2 = r2_score(y_full, y_full_pred)
full_mae = mean_absolute_error(y_full, y_full_pred)
print(f"Full dataset R2:  {full_r2:.4f}")
print(f"Full dataset MAE: {full_mae:.4f}")

---
## 10. Summary

| Item | Value |
|------|-------|
| **Model** | Linear Regression (scikit-learn) |
| **Features** | timestamp, requests, cpu_usage |
| **Target** | Next time-step request count |
| **Train/Test Split** | 80% / 20% |
| **R-squared** | ~0.9717 |
| **MAE** | ~12.71 |
| **Model File** | `models/trained_model.pkl` |
| **Serialization** | Joblib |

The model is ready for deployment in the auto-scaling dashboard (`app.py`).